# PKG staging v2: name-confirmed duplicate rule, 5C/6C dedup map, counterparty-id crosswalk

v1 found that 5C↔6C share a same-day (account, amount) key 19–21× above placebo, with `cpty_name` equal in 99.94% of pairs.
v2 turns that into a rule and tests the internal-vs-external hypothesis.

**Other-party name.** Every leg carries the name of the party on the other side of the PNC account:
- external row → `cpty_name`
- internal row, anchored on the receiver → `customer_name_pays`
- internal row, anchored on the payer → `customer_name_receives`

One key, `(dt, pnc_acct, role, amount, other_name)`, therefore tests 5C↔6C (cpty vs cpty) and 7A↔7C
(customer vs cpty) with the same code. That is the exact `cpty_name` ↔ `customer_name` match.

| § | What | Output |
|---|---|---|
| 1 | Every configured category pair: amount-only vs **name-confirmed** match, placebo ±7d, lift, verdict | 2 tables |
| 2 | 5C/6C **dedup map**: explicit one-to-one pairing; tier 1 same day, tier 2 6C one day later | 3 tables |
| 3 | **Counterparty-id crosswalk** 6C→5C: stability, fan-out, how much of unmatched 6C inherits a 5C id + FI | 6 tables |

Nothing is written unless `WRITE_OUTPUTS = True`.

In [ ]:
# ---- Config -------------------------------------------------------------------------------
TABLE    = "dsihd01p_dsi.neo4j_payments"
START_DT = "2024-10-01"
END_DT   = "2025-01-31"            # inclusive; 4 whole months

KEEP_CAT = "5C.RTP_PRT_CPTY_PAYS"  # kept record: carries the FI
DROP_CAT = "6C.RTP_P2P_CPTY_PAYS"  # flagged record

# (A, B) pairs. Names are resolved case-insensitively against the partitions; a miss is reported and skipped.
PAIRS = [
    ("5C.RTP_PRT_CPTY_PAYS",                   "6C.RTP_P2P_CPTY_PAYS"),                   # reference
    ("7A.CHECK_PAYS",                          "7C.CHECK_CPTY_PAYS"),
    ("2A.ACH_OrigViaPNC_woTPO_PAYS",           "2C.ACH_OrigViaPNC_woTPO_CPTY_PAYS"),
    ("2A.ACH_OrigViaPNC_woTPO_PAYS",           "2B.ACH_OrigViaPNC_woTPO_PAYS_CPTY"),
    ("6A.RTP_P2P_PAYS",                        "6C.RTP_P2P_CPTY_PAYS"),
    ("6A.RTP_P2P_PAYS",                        "6B.RTP_P2P_PAYS_CPTY"),
    ("5C.RTP_PRT_CPTY_PAYS",                   "6A.RTP_P2P_PAYS"),
    ("1A1.ACH_OrigViaPNC_wTPO_ORIG_PAYS_CUST", "1B.ACH_OrigViaPNC_wTPO_ORIG_PAYS_CPTY"),
    ("1A2.ACH_OrigViaPNC_wTPO_CUST_PAYS_ORIG", "1C.ACH_OrigViaPNC_wTPO_CPTY_PAYS_ORIG"),
    ("1A2.ACH_OrigViaPNC_wTPO_CUST_PAYS_ORIG", "3B.ACH_OrigViaNONPNC_PAYS_CPTY"),
    ("5A.RTP_PRT_PAYS",                        "5B.RTP_PRT_PAYS_CPTY"),                   # control: expect ~1
]
LAGS = [0, 1, -1, 7, -7]           # lag = B.dt − A.dt.  +1 = B stamped a day later.  ±7 = placebo

# Verdict thresholds on the NAME-confirmed lift (same-day ÷ placebo)
LIFT_DUP, LIFT_PARTIAL = 10.0, 3.0
MIN_EXCESS_ROWS        = 1_000     # excess = same-day dup rows − placebo dup rows
NULL_NAME_MAX          = 0.50      # above this share of unnamed legs on either side, the name test can't speak

WRITE_OUTPUTS = False              # True writes the two tables below (overwrite)
DUP_MAP_TABLE = "dsihd01p_dsi.pkg_stg_dupmap_5c6c"
XWALK_TABLE   = "dsihd01p_dsi.pkg_stg_cptyxwalk_6c5c"

In [ ]:
# ---- NumPy compat shim (must run before any toPandas) ------------------------------------
# Cloudera SPARK3's PySpark references NumPy aliases removed in 1.24/2.0; toPandas() fails on
# empty results and on the Arrow path with "module 'numpy' has no attribute 'object0'".
import numpy as np, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
warnings.filterwarnings("ignore", category=ResourceWarning)                       # py4j socket noise
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")  # distutils LooseVersion noise

from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display
from urllib.parse import unquote
from difflib import get_close_matches
import pandas as pd, re, time, datetime as dt

spark = SparkSession.builder.getOrCreate()
print("Spark", spark.version, "| pandas", pd.__version__, "| numpy", np.__version__)
MD = StorageLevel.MEMORY_AND_DISK

pd.set_option("display.max_rows", 200); pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250);    pd.set_option("display.max_colwidth", 80)

def show(pdf, title, pct=(), money=()):
    print(f"\n■ {title}   [{len(pdf):,} rows]")
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                fmt[c] = "{:.2%}"
        elif c in money:                            fmt[c] = "${:,.0f}"
        elif pd.api.types.is_integer_dtype(pdf[c]): fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):   fmt[c] = "{:,.2f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:    sty = sty.hide(axis="index")
    except AttributeError: sty = sty.hide_index()
    display(sty)

def ints(pdf, cols):
    for c in cols:
        if c in pdf.columns: pdf[c] = pdf[c].fillna(0).astype("int64")
    return pdf

## §0 Schema, category resolution, window

In [ ]:
_db, _tbl = TABLE.split(".", 1) if "." in TABLE else (spark.catalog.currentDatabase(), TABLE)
try:    _catalog = spark.catalog.listColumns(f"{_db}.{_tbl}")
except Exception: _catalog = spark.catalog.listColumns(_tbl, _db)
COLTYPE   = {c.name.lower(): c.dataType.lower() for c in _catalog}
PART_COLS = [c.name.lower() for c in _catalog if c.isPartition]

REQUIRED = ["trans_id", "trans_dt", "trans_amt", "category", "cpty_type",
            "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_pays", "pnc_dep_acct_receives",
            "customer_name_pays", "customer_name_receives",
            "unq_cpty_acct_id", "cpty_name", "cpty_fin_entity_name"]
missing = [c for c in REQUIRED if c not in COLTYPE]
assert not missing, f"Missing columns in {TABLE}: {missing}"
assert "category" in PART_COLS, "v2 relies on category partition pruning; category is not a partition column"

# Resolve configured names against the metastore (no data scanned)
CATS_ACTUAL = set()
for r in spark.sql(f"SHOW PARTITIONS {TABLE}").collect():
    kv = {k.lower(): unquote(v) for k, v in (p.split("=", 1) for p in r[0].split("/"))}
    CATS_ACTUAL.add(kv["category"])
_up = {c.upper(): c for c in CATS_ACTUAL}
resolve = lambda c: _up.get(c.upper())

PAIRS_OK = []
for a, b in PAIRS:
    ra, rb = resolve(a), resolve(b)
    if ra and rb:
        PAIRS_OK.append((ra, rb))
    for c, r in ((a, ra), (b, rb)):
        if not r:
            print(f"⚠ not found, pair skipped: {c}   closest: {get_close_matches(c, sorted(CATS_ACTUAL), 3, 0.6)}")
_k, _d = resolve(KEEP_CAT), resolve(DROP_CAT)
assert _k and _d, f"KEEP_CAT / DROP_CAT not in partitions: {KEEP_CAT}, {DROP_CAT}"
KEEP_CAT, DROP_CAT = _k, _d
print(f"{len(PAIRS_OK)} of {len(PAIRS)} pairs resolved | window {START_DT} → {END_DT}")

def range_on(col, typ, lo, hi, fmt="%Y-%m-%d"):
    if typ == "date":
        return F.col(col).between(F.lit(lo).cast("date"), F.lit(hi).cast("date"))
    if typ.startswith("timestamp"):
        return (F.col(col) >= F.lit(lo).cast("timestamp")) & (F.col(col) < F.date_add(F.lit(hi).cast("date"), 1).cast("timestamp"))
    return F.col(col).between(dt.date.fromisoformat(lo).strftime(fmt), dt.date.fromisoformat(hi).strftime(fmt))

DT_COND = range_on("trans_dt", COLTYPE["trans_dt"], START_DT, END_DT)

def src(cats):
    """Window + category predicates on the two partition columns: reads only those partitions."""
    return spark.table(TABLE).filter(DT_COND & F.col("category").isin(list(cats)))

_plan  = src([KEEP_CAT])._jdf.queryExecution().executedPlan().toString()
_pf    = re.findall(r"PartitionFilters: \[([^\]]*)\]", _plan)
_paths = re.findall(r"FileIndex[^(]*\((\d+) paths\)", _plan)
print("PartitionFilters:", (_pf[0][:300] if _pf else "n/a"), "| paths read for KEEP_CAT:", (_paths[0] if _paths else "n/a"))
if _pf and "category" not in _pf[0]:
    print("⚠ category predicate is not pruning partitions")

In [ ]:
# ---- Legs with the other party's name ----------------------------------------------------------
def nz(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

def shape(c):
    """Masked format: '-'-segments → D(igits)/A(lpha)/X(mixed)/E(mpty) + length."""
    return F.expr(f"""CASE WHEN {c} IS NULL THEN 'NULL' ELSE array_join(transform(split({c}, '-'), x -> concat(
        CASE WHEN x = '' THEN 'E' WHEN x rlike '^[0-9]+$' THEN 'D' WHEN x rlike '^[A-Za-z]+$' THEN 'A' ELSE 'X' END,
        CAST(length(x) AS STRING))), '-') END""")

# exact  = case- and whitespace-insensitive, otherwise literal  (the rule)
# loose  = letters+digits only                                  (diagnostic: how much punctuation costs)
nm_exact = lambda c: F.upper(F.regexp_replace(c, r"\s+", " "))
def nm_loose(c):
    s = F.regexp_replace(F.upper(c), "[^A-Z0-9]", "")
    return F.when(s == "", F.lit(None)).otherwise(s)

KEY = ["pnc_acct", "role", "amt_key"]

def legs_for(cats):
    b = (src(cats).select(
            nz("trans_id").alias("trans_id"), F.to_date(F.col("trans_dt")).alias("dt"),
            F.abs(F.col("trans_amt").cast("decimal(22,2)")).alias("amt_key"),   # exact cents
            nz("category").alias("category"),
            nz("mdm_id_pays").alias("mdm_pays"), nz("mdm_id_receives").alias("mdm_recv"),
            nz("pnc_dep_acct_pays").alias("acct_pays"), nz("pnc_dep_acct_receives").alias("acct_recv"),
            nz("customer_name_pays").alias("cust_pays"), nz("customer_name_receives").alias("cust_recv"),
            nz("cpty_name").alias("cpty_name"), nz("unq_cpty_acct_id").alias("cpty_id"),
            nz("cpty_fin_entity_name").alias("cpty_fi"), nz("cpty_type").alias("cpty_type"))
         .withColumn("internal", F.col("mdm_pays").isNotNull() & F.col("mdm_recv").isNotNull()))
    # One leg per PNC account on the row. other_name = whoever sits on the far side of that account.
    arr = F.array(
        F.when(F.col("acct_pays").isNotNull(), F.struct(
            F.lit("PAYER").alias("role"), F.col("acct_pays").alias("pnc_acct"),
            F.when(F.col("internal"), F.col("cust_recv")).otherwise(F.col("cpty_name")).alias("other_name"))),
        F.when(F.col("acct_recv").isNotNull(), F.struct(
            F.lit("RECEIVER").alias("role"), F.col("acct_recv").alias("pnc_acct"),
            F.when(F.col("internal"), F.col("cust_pays")).otherwise(F.col("cpty_name")).alias("other_name"))))
    return (b.withColumn("leg", F.explode(arr)).filter(F.col("leg").isNotNull())
             .select("*", "leg.role", "leg.pnc_acct", "leg.other_name").drop("leg")
             .withColumn("nm_x", nm_exact(F.col("other_name")))
             .withColumn("nm_l", nm_loose(F.col("other_name")))
             .withColumn("amt", F.col("amt_key").cast("double")))

## §1 Every pair: amount-only vs name-confirmed

How to read the columns:
- `name_share_of_amt` is the fraction of amount-only same-day matches whose other-party names also agree. It is high for a true duplicate and near the placebo level for a coincidence.
- `name_lift` is the verdict driver.
- `B_later_1d` / `B_earlier_1d` measure date skew under the name key; compare them with `name_placebo`.
- `excess_dup_rows` is same-day duplicate rows minus the placebo expectation: the rows a rule would remove *beyond chance*.

In [ ]:
FINE, ROLES = {}, {}
def fine(cat):
    """Per-category aggregate at (dt, acct, role, amount, names). One partition scan per category, reused across pairs."""
    if cat not in FINE:
        FINE[cat] = (legs_for([cat]).groupBy("dt", *KEY, "nm_x", "nm_l")
                       .agg(F.count("*").alias("n")).persist(MD))
    return FINE[cat]

def roles(cat):
    if cat not in ROLES:
        ROLES[cat] = {r[0] for r in fine(cat).select("role").distinct().collect()}
    return ROLES[cat]

VARIANTS = {"amt": ([], [0, 7, -7]), "name": (["nm_x"], LAGS), "loose": (["nm_l"], [0, 7, -7])}

def lag_table(fa, fb, cols, lags):
    ka = fa.groupBy("dt", *KEY, *cols).agg(F.sum("n").alias("n_a"))
    kb = fb.groupBy("dt", *KEY, *cols).agg(F.sum("n").alias("n_b"))
    # Only (acct, role, amount) present on both sides can match at any lag: prune before the lag fan-out
    ka = ka.join(kb.select(*KEY).distinct(), KEY, "left_semi")
    kb = kb.join(ka.select(*KEY).distinct(), KEY, "left_semi")
    lag_df = spark.createDataFrame([(int(l),) for l in lags], "lag int")   # date_sub needs INT, not BIGINT
    kb = kb.crossJoin(F.broadcast(lag_df)).withColumn("dt", F.expr("date_sub(dt, lag)"))   # B.dt = A.dt + lag
    out = (ka.join(kb, ["dt", *KEY, *cols])                                                 # null name never joins
             .groupBy("lag").agg(F.sum("n_a").alias("a_rows"), F.sum("n_b").alias("b_rows"),
                                 F.sum(F.least("n_a", "n_b")).alias("dup_rows"),
                                 F.sum(F.least("n_a", "n_b") * F.col("amt_key").cast("double")).alias("dup_usd"))
             .toPandas())
    return out.set_index("lag").reindex(lags).fillna(0)

T1, T2, RES = [], [], {}
for a, b in PAIRS_OK:
    t0 = time.time()
    rs = sorted(roles(a) & roles(b))
    code_ = lambda c: c.split(".")[0]
    label = f"{code_(a)} ↔ {code_(b)}"
    if not rs:
        print(f"skip {label}: no shared PNC role"); continue
    fa, fb = fine(a).filter(F.col("role").isin(rs)), fine(b).filter(F.col("role").isin(rs))
    tot = lambda f: f.agg(F.sum("n").alias("rows"),
                          F.sum(F.when(F.col("nm_x").isNull(), F.col("n")).otherwise(0)).alias("null_nm")).first()
    ta, tb = tot(fa), tot(fb)
    nA, nB = int(ta["rows"] or 0), int(tb["rows"] or 0)
    res = {v: lag_table(fa, fb, cols, lags) for v, (cols, lags) in VARIANTS.items()}
    RES[(a, b)] = res

    pa   = lambda v, l: res[v].loc[l, "a_rows"] / max(nA, 1)
    plc  = lambda v: (pa(v, 7) + pa(v, -7)) / 2
    lift = lambda v: (pa(v, 0) / plc(v)) if plc(v) > 0 else (float("inf") if pa(v, 0) > 0 else float("nan"))
    dup0     = int(res["name"].loc[0, "dup_rows"])
    dup_plc  = (res["name"].loc[7, "dup_rows"] + res["name"].loc[-7, "dup_rows"]) / 2
    excess   = dup0 - dup_plc
    null_sh  = max((ta["null_nm"] or 0) / max(nA, 1), (tb["null_nm"] or 0) / max(nB, 1))
    nl = lift("name")
    verdict = ("NAMES MISSING"   if null_sh > NULL_NAME_MAX else
               "DUPLICATE"       if nl >= LIFT_DUP and excess >= MIN_EXCESS_ROWS else
               "PARTIAL / REVIEW" if nl >= LIFT_PARTIAL and excess >= MIN_EXCESS_ROWS else
               "COINCIDENCE")

    T1.append({"pair": label, "anchor": "/".join(rs), "rows_A": nA, "rows_B": nB,
               "amt_same_day": pa("amt", 0), "amt_placebo": plc("amt"), "amt_lift": lift("amt"),
               "name_same_day": pa("name", 0), "name_placebo": plc("name"), "name_lift": nl,
               "name_share_of_amt": pa("name", 0) / pa("amt", 0) if pa("amt", 0) > 0 else float("nan"),
               "verdict": verdict})
    T2.append({"pair": label, "dup_rows": dup0, "placebo_dup_rows": int(round(dup_plc)),
               "excess_dup_rows": int(round(excess)), "excess_pct_of_B": excess / max(nB, 1),
               "dup_usd": res["name"].loc[0, "dup_usd"],
               "loose_same_day": pa("loose", 0), "B_later_1d": pa("name", 1), "B_earlier_1d": pa("name", -1),
               "null_name_A": (ta["null_nm"] or 0) / max(nA, 1), "null_name_B": (tb["null_nm"] or 0) / max(nB, 1)})
    print(f"  {label:<14} {verdict:<17} name lift {nl:,.1f}   ({time.time() - t0:,.0f}s)")

t1 = pd.DataFrame(T1); t2 = pd.DataFrame(T2)
show(t1, "Match rates, share of A rows (anchor = PNC role shared by both categories)",
     pct=("amt_same_day", "amt_placebo", "name_same_day", "name_placebo", "name_share_of_amt"))
show(t2, "Sizing and diagnostics (name-confirmed key)",
     pct=("excess_pct_of_B", "loose_same_day", "B_later_1d", "B_earlier_1d", "null_name_A", "null_name_B"),
     money=("dup_usd",))

for f in FINE.values(): f.unpersist()
FINE.clear()

## §2 5C/6C dedup map

This is an explicit one-to-one pairing, so every flagged 6C row points at the exact 5C row it duplicates.
- **Tier 1:** same day, same `(acct, role, amount, other_name)`. Within a key group the k-th 5C row pairs with the k-th 6C row, ordered by `trans_id`.
- **Tier 2:** 6C rows left over from tier 1, paired with leftover 5C rows one day *earlier*. v1 showed this skew runs in one direction only.

Legs with no counterparty name cannot be paired under this rule; they are counted, not guessed.

In [ ]:
anchor = sorted(roles(KEEP_CAT) & roles(DROP_CAT))
assert anchor, "KEEP_CAT and DROP_CAT share no PNC role"
rows = (legs_for([KEEP_CAT, DROP_CAT]).filter(F.col("role").isin(anchor))
          .select("trans_id", "dt", "category", *KEY, "amt", "nm_x", "cpty_id", "cpty_fi", "cpty_type",
                  shape("cpty_id").alias("cpty_id_shape"))
          .withColumn("month", F.date_format("dt", "yyyy-MM"))
          .persist(MD))
A = rows.filter(F.col("category") == KEEP_CAT)
B = rows.filter(F.col("category") == DROP_CAT)

GK = ["dt", *KEY, "nm_x"]
CARRY = ["trans_id", "cpty_id", "cpty_fi", "cpty_type", "amt", "month"]
def ranked(df, sfx):
    w = Window.partitionBy(*GK).orderBy("trans_id")
    return (df.filter(F.col("nm_x").isNotNull()).withColumn("rn", F.row_number().over(w))
              .select(*GK, "rn", *[F.col(c).alias(f"{c}_{sfx}") for c in CARRY]))

t1 = (ranked(A, "a").join(ranked(B, "b"), GK + ["rn"])
        .withColumn("tier", F.lit(1)).withColumn("lag_days", F.lit(0)))
A_left = A.join(t1.select(F.col("trans_id_a").alias("trans_id")), "trans_id", "left_anti")
B_left = B.join(t1.select(F.col("trans_id_b").alias("trans_id")), "trans_id", "left_anti")
t2 = (ranked(A_left, "a").join(ranked(B_left.withColumn("dt", F.date_sub("dt", 1)), "b"), GK + ["rn"])
        .withColumn("tier", F.lit(2)).withColumn("lag_days", F.lit(1)))
dup_map = t1.unionByName(t2).persist(MD)

side = (rows.groupBy("category").agg(F.count("*").alias("rows"), F.sum("amt").alias("usd"),
                                     F.sum(F.col("nm_x").isNull().cast("long")).alias("no_name_rows"))
            .toPandas().set_index("category"))
nK, nD = int(side.loc[KEEP_CAT, "rows"]), int(side.loc[DROP_CAT, "rows"])
uD = float(side.loc[DROP_CAT, "usd"])

tiers = (dup_map.groupBy("tier", "lag_days").agg(F.count("*").alias("pairs"), F.sum("amt_b").alias("usd"))
                .orderBy("tier").toPandas())
tiers = pd.concat([tiers, pd.DataFrame([{"tier": "total", "lag_days": None,
                                         "pairs": tiers.pairs.sum(), "usd": tiers.usd.sum()}])], ignore_index=True)
tiers["pct_of_DROP_rows"] = tiers.pairs / nD
tiers["pct_of_DROP_usd"]  = tiers.usd / uD
tiers["pct_of_KEEP_rows"] = tiers.pairs / nK
show(ints(tiers, ["pairs"]), f"Dedup map: {DROP_CAT} rows flagged, each paired to one {KEEP_CAT} row",
     pct=("pct_of_DROP_rows", "pct_of_DROP_usd", "pct_of_KEEP_rows"), money=("usd",))

side_pdf = side.reset_index()
side_pdf["no_name_share"] = side_pdf.no_name_rows / side_pdf.rows
show(ints(side_pdf, ["rows", "no_name_rows"]), "Rows per side; unnamed rows cannot be paired under the name rule",
     pct=("no_name_share",), money=("usd",))

bym = (B.groupBy("month").count().withColumnRenamed("count", "drop_rows")
        .join(dup_map.groupBy(F.col("month_b").alias("month")).pivot("tier", [1, 2]).count(), "month", "left")
        .orderBy("month").toPandas())
bym = ints(bym, ["drop_rows", "1", "2"]).rename(columns={"1": "tier1", "2": "tier2"})
bym["flagged_share"] = (bym.tier1 + bym.tier2) / bym.drop_rows
show(bym, "Flagged share of DROP_CAT by month (stability check)", pct=("flagged_share",))

## §3 Counterparty-id crosswalk 6C → 5C

Every pair in the map ties a 6C `unq_cpty_acct_id` to a 5C `unq_cpty_acct_id`. The crosswalk is only useful if 6C ids are
**persistent**, meaning one id recurs across many payments. If most 6C ids appear once, the id is closer to a per-transaction
token, and unmatched 6C rows cannot inherit anything. The first table answers that question.

In [ ]:
fld = (rows.groupBy("category").agg(
          F.count("*").alias("rows"),
          F.expr("approx_count_distinct(cpty_id)").alias("distinct_cpty_ids"),
          F.avg(F.col("cpty_id").isNull().cast("double")).alias("cpty_id_null"),
          F.avg(F.col("cpty_fi").isNull().cast("double")).alias("cpty_fi_null"),
          F.avg(F.col("nm_x").isNull().cast("double")).alias("cpty_name_null"),
          F.avg(F.col("cpty_type").isNull().cast("double")).alias("cpty_type_null"))
         .toPandas())
fld["rows_per_cpty_id"] = fld.rows / fld.distinct_cpty_ids.clip(lower=1)
show(ints(fld, ["rows", "distinct_cpty_ids"]), "Field coverage, all rows in window (distinct ids approximate)",
     pct=("cpty_id_null", "cpty_fi_null", "cpty_name_null", "cpty_type_null"))

shp = (rows.groupBy("category", "cpty_id_shape").count()
           .withColumn("share", F.col("count") / F.sum("count").over(Window.partitionBy("category")))
           .orderBy("category", F.desc("count")).toPandas())
show(shp.groupby("category").head(5), "unq_cpty_acct_id format per category (masked, top 5)", pct=("share",))

ct = (dup_map.groupBy(F.col("cpty_type_a").alias(f"cpty_type {KEEP_CAT[:2]}"),
                      F.col("cpty_type_b").alias(f"cpty_type {DROP_CAT[:2]}")).count()
             .orderBy(F.desc("count")).limit(12).toPandas())
ct["share"] = ct["count"] / ct["count"].sum()
show(ct, "cpty_type values across paired rows (top 12)", pct=("share",))

In [ ]:
xw = (dup_map.filter(F.col("cpty_id_a").isNotNull() & F.col("cpty_id_b").isNotNull())
             .groupBy("cpty_id_b", "cpty_id_a")
             .agg(F.count("*").alias("pairs"), F.max("cpty_fi_a").alias("fi_a"))
             .persist(MD))

per_b = xw.groupBy("cpty_id_b").agg(F.sum("pairs").alias("pairs"), F.count("*").alias("n_a_ids"),
                                    F.max("pairs").alias("top_pairs"))
bk = lambda c, edges: F.when(F.col(c) <= edges[0], f"{edges[0]}").when(F.col(c) <= edges[1], f"{edges[0]+1}–{edges[1]}") \
                       .when(F.col(c) <= edges[2], f"{edges[1]+1}–{edges[2]}").otherwise(f"{edges[2]+1}+")
persist_tbl = (per_b.withColumn("pairs_per_6C_id", bk("pairs", (1, 5, 20)))
                    .groupBy("pairs_per_6C_id").agg(F.count("*").alias("ids"), F.sum("pairs").alias("pairs"))
                    .toPandas())
persist_tbl["id_share"]   = persist_tbl.ids / persist_tbl.ids.sum()
persist_tbl["pair_share"] = persist_tbl.pairs / persist_tbl.pairs.sum()
persist_tbl["_o"] = persist_tbl.pairs_per_6C_id.map({"1": 0, "2–5": 1, "6–20": 2, "21+": 3})
show(ints(persist_tbl.sort_values("_o").drop(columns="_o"), ["ids", "pairs"]),
     "Persistence: paired payments per DROP_CAT cpty_id", pct=("id_share", "pair_share"))

fan_b = (per_b.withColumn("keep_ids_per_drop_id", F.when(F.col("n_a_ids") >= 3, "3+").otherwise(F.col("n_a_ids").cast("string")))
              .groupBy("keep_ids_per_drop_id")
              .agg(F.count("*").alias("drop_ids"), F.sum("pairs").alias("pairs"),
                   F.avg(F.col("top_pairs") / F.col("pairs")).alias("mean_purity"))
              .orderBy("keep_ids_per_drop_id").toPandas())
fan_a = (xw.groupBy("cpty_id_a").agg(F.count("*").alias("n_b"))
           .withColumn("drop_ids_per_keep_id", F.when(F.col("n_b") >= 3, "3+").otherwise(F.col("n_b").cast("string")))
           .groupBy("drop_ids_per_keep_id").agg(F.count("*").alias("keep_ids")).orderBy("drop_ids_per_keep_id").toPandas())
show(ints(fan_b, ["drop_ids", "pairs"]), "Fan-out: KEEP ids per DROP id (purity = top KEEP id's share of that DROP id's pairs)",
     pct=("mean_purity",))
show(ints(fan_a, ["keep_ids"]), "Fan-out: DROP ids per KEEP id")

# Resolved crosswalk: each DROP id → its most frequent KEEP id
w_best = Window.partitionBy("cpty_id_b").orderBy(F.desc("pairs"), "cpty_id_a")
xw_best = (xw.withColumn("r", F.row_number().over(w_best)).filter("r = 1").drop("r")
             .join(per_b.select("cpty_id_b", F.col("pairs").alias("pairs_total")), "cpty_id_b")
             .withColumn("purity", F.col("pairs") / F.col("pairs_total"))
             .persist(MD))

# What the crosswalk buys: DROP rows that are NOT duplicates, whose cpty_id the crosswalk knows
B_rem = B.join(dup_map.select(F.col("trans_id_b").alias("trans_id")), "trans_id", "left_anti")
cov = (B_rem.join(xw_best.select(F.col("cpty_id_b").alias("cpty_id"), "fi_a", F.lit(1).alias("in_xw")), "cpty_id", "left")
            .agg(F.count("*").alias("rows"), F.sum("amt").alias("usd"),
                 F.sum(F.col("in_xw").isNotNull().cast("long")).alias("rows_in_xw"),
                 F.sum(F.when(F.col("in_xw").isNotNull(), F.col("amt")).otherwise(0)).alias("usd_in_xw"),
                 F.sum((F.col("in_xw").isNotNull() & F.col("fi_a").isNotNull()).cast("long")).alias("rows_fi_inheritable"),
                 F.sum(F.when(F.col("in_xw").isNotNull() & F.col("fi_a").isNotNull(), F.col("amt")).otherwise(0)).alias("usd_fi_inheritable"),
                 F.sum(F.col("cpty_fi").isNotNull().cast("long")).alias("rows_with_own_fi"))
            .toPandas())
c = cov.iloc[0]
cov_tbl = pd.DataFrame([
    {"metric": f"{DROP_CAT} rows NOT flagged as duplicates", "rows": c.rows, "pct_rows": 1.0, "usd": c.usd, "pct_usd": 1.0},
    {"metric": "… cpty_id present in crosswalk (inherits KEEP id)", "rows": c.rows_in_xw,
     "pct_rows": c.rows_in_xw / max(c.rows, 1), "usd": c.usd_in_xw, "pct_usd": c.usd_in_xw / max(c.usd, 1)},
    {"metric": "… and the KEEP id carries an FI (inherits FI)", "rows": c.rows_fi_inheritable,
     "pct_rows": c.rows_fi_inheritable / max(c.rows, 1), "usd": c.usd_fi_inheritable, "pct_usd": c.usd_fi_inheritable / max(c.usd, 1)},
    {"metric": "… already carrying their own FI (baseline)", "rows": c.rows_with_own_fi,
     "pct_rows": c.rows_with_own_fi / max(c.rows, 1), "usd": None, "pct_usd": None},
])
show(ints(cov_tbl, ["rows"]), "Crosswalk coverage of unmatched DROP_CAT rows", pct=("pct_rows", "pct_usd"), money=("usd",))

In [ ]:
if WRITE_OUTPUTS:
    (dup_map.select(F.col("trans_id_b").alias("trans_id_drop"), F.col("trans_id_a").alias("trans_id_keep"),
                    "tier", "lag_days", F.col("dt").alias("dt_keep"), "amt_key")
            .write.mode("overwrite").saveAsTable(DUP_MAP_TABLE))
    (xw_best.select(F.col("cpty_id_b").alias("cpty_id_drop"), F.col("cpty_id_a").alias("cpty_id_keep"),
                    F.col("fi_a").alias("cpty_fi_keep"), "pairs", "pairs_total", "purity")
            .write.mode("overwrite").saveAsTable(XWALK_TABLE))
    print("written:", DUP_MAP_TABLE, XWALK_TABLE)
else:
    print("WRITE_OUTPUTS = False: nothing written")

for _df in ["xw_best", "xw", "dup_map", "rows"]:
    if _df in globals(): globals()[_df].unpersist()